In [7]:
#Source : https://github.com/PacktPublishing/Python-Natural-Language-Processing-Cookbook-Second-Edition/tree/main/data

In [8]:
# !python -m spacy download en_core_web_sm

# Simple classifier

In [9]:
import nltk
import spacy
from nltk import word_tokenize
from nltk.corpus import stopwords
from string import punctuation
small_model = spacy.load("en_core_web_sm")
large_model = spacy.load("en_core_web_lg")

In [10]:
from datasets import load_dataset
train_dataset = load_dataset("rotten_tomatoes", split="train[:15%]+train[-15%:]")
test_dataset = load_dataset("rotten_tomatoes", split="test[:15%]+test[-15%:]")

/Users/lwhitenack/CS-620/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 1066/1066 [00:00<00:00, 563185.30 examples/s]


In [11]:
print(len(train_dataset))
print(len(test_dataset))

2560
320


In [12]:
class POS_vectorizer:
    def __init__(self, spacy_model):
        self.model = spacy_model

    def vectorize(self, input_text):
        doc = self.model(input_text)
        vector = []
        vector.append(len(doc))
        pos = {"VERB":0, "NOUN":0, "PROPN":0, "ADJ":0, "ADV":0, "AUX":0, "PRON":0, "NUM":0, "PUNCT":0}
        for token in doc:
            if token.pos_ in pos.keys():
                pos[token.pos_] += 1
        vector_values = list(pos.values())
        vector = vector + vector_values
        return vector

In [13]:
sample_text = train_dataset[0]["text"]
vectorizer = POS_vectorizer(small_model)
vector = vectorizer.vectorize(sample_text)

In [14]:
print(sample_text)
print(vector)

the rock is destined to be the 21st century's new " conan " and that he's going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .
[38, 3, 8, 2, 5, 1, 3, 1, 0, 5]


In [15]:
import pandas as pd
import numpy as np
train_df = train_dataset.to_pandas()
train_df.sample(frac=1)
test_df = test_dataset.to_pandas()
train_df["vector"] = train_df["text"].apply(lambda x: vectorizer.vectorize(x))
test_df["vector"] = test_df["text"].apply(lambda x: vectorizer.vectorize(x))
X_train = np.stack(train_df["vector"].values, axis=0)
X_test = np.stack(test_df["vector"].values, axis=0)
y_train = train_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
clf = LogisticRegression(C=0.1)
clf = clf.fit(X_train, y_train)

/Users/lwhitenack/CS-620/.venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/lwhitenack/CS-620/.venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/lwhitenack/CS-620/.venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/lwhitenack/CS-620/.venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/lwhitenack/CS-620/.venv/lib/python3.13/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: overflow encountered in matmul
  grad[:n_features] = X.T @ grad_point

In [17]:
test_df["prediction"] = test_df["vector"].apply(lambda x: clf.predict([x])[0])
print(classification_report(test_df["label"], test_df["prediction"]))

              precision    recall  f1-score   support

           0       0.58      0.56      0.57       160
           1       0.58      0.59      0.58       160

    accuracy                           0.58       320
   macro avg       0.58      0.58      0.58       320
weighted avg       0.58      0.58      0.58       320



In [18]:
# Get a sample review from the test dataset
sample_review = test_dataset[0]["text"]

# Vectorize the sample review
vectorized_review = vectorizer.vectorize(sample_review)

# Print the review and its vectorization
print("Sample Review:")
print(sample_review)
print("\nPOS Vectorization:")
print(vectorized_review)

Sample Review:
lovingly photographed in the manner of a golden book sprung to life , stuart little 2 manages sweetness largely without stickiness .

POS Vectorization:
[22, 3, 5, 1, 2, 2, 0, 0, 1, 2]
